# 01 — Batch Processing

Run the full pipeline (resolve → consolidate → visualize) across all
protocols in a collection.

## Configuration

In [ ]:
#!pip install markdown

In [ ]:
import sys
from pathlib import Path

# Make the repo-root package importable (or `pip install -e .` from the repo root)
sys.path.insert(0, str(Path.cwd().parent))

from soa2usdm import config
from soa2usdm.corrections import ApplyCorrectionsStep
from soa2usdm.resolve import ResolveStep
from soa2usdm.consolidate import ConsolidateStep
from soa2usdm.visualize import VisualizeStep
from soa2usdm.visualize_resolved import VisualizeResolvedStep
from soa2usdm.index_generator import IndexGeneratorStep
from soa2usdm.errors import Errors
from soa2usdm.analytics import Analytics

# === CONFIGURE ===
COLLECTION = 'usdm_data'
ONLY_PROTOCOLS = []  # empty = all, or e.g. ['NCT03637764', 'NCT04677179']

protocols = config.list_extractable_protocols(COLLECTION)
if ONLY_PROTOCOLS:
    protocols = [p for p in protocols if p in ONLY_PROTOCOLS]

all_protocols = config.list_protocols(COLLECTION)
skipped = len(all_protocols) - len(protocols)

print(f'Collection: {COLLECTION}')
print(f'Protocols:  {protocols}')
if skipped:
    print(f'Skipped:    {skipped} protocol(s) without extraction files')

## Run pipeline

In [ ]:
STEPS = [ApplyCorrectionsStep, ResolveStep, VisualizeResolvedStep, ConsolidateStep, VisualizeStep]

results = {}

for protocol_id in protocols:
    print(f'\n{"=" * 60}')
    print(f'{protocol_id}')
    print(f'{"=" * 60}')

    errors = Errors()
    analytics = Analytics()
    source = {'protocol_id': protocol_id, 'collection': COLLECTION}
    data = {'source': source}

    for step_class in STEPS:
        step = step_class(errors, analytics)
        step_name = step_class.step_name
        data[step_name] = step.execute(data)
        print(f'  {step_name}: done')

    if errors.has_errors():
        print(f'  ⚠ {errors.count()} error(s):')
        for e in errors.all:
            print(f'    [{e.step}] {e.message}')

    results[protocol_id] = {
        'data': data,
        'errors': errors,
        'analytics': analytics,
    }

print(f'\n{"=" * 60}')
print(f'Done: {len(results)} protocols')

## Summary

In [ ]:
print(f'{"Protocol":<20} {"Tables":>6} {"Activities":>10} {"Compress":>8} {"Status":>8}')
print('-' * 60)

for pid, r in results.items():
    cons = r['data'].get('consolidate', {})
    status = '✗' if r['errors'].has_errors() else '✓'
    print(f'{pid:<20} '
          f'{cons.get("tables", "?"):>6} '
          f'{cons.get("unified_activities", "?"):>10} '
          f'{str(cons.get("compression_percent", "?")) + "%":>8} '
          f'{status:>8}')

## Collection Index

In [ ]:
# Generate collection index.html
idx_errors = Errors()
idx_analytics = Analytics()
idx_step = IndexGeneratorStep(idx_errors, idx_analytics)

idx_result = idx_step.execute({'source': {'collection': COLLECTION}})

if idx_result.get('status') == 'success':
    print(f'Index: {idx_result["output_file"]}')
else:
    print(f'Index generation failed: {idx_result.get("error", "unknown")}')